| Tabela Gold               | Granularidade   | Finalidade                           |
| ------------------------- | --------------- | ------------------------------------ |
| `cobertura_municipio`     | Município × Ano | Comparações e distribuição municipal |
| `cobertura_uf`            | UF × Ano        | Evolução oficial por estado          |
| `cobertura_regiao`        | Região × Ano    | Evolução oficial por região          |
| `cobertura_pib_municipio` | Município × Ano | Relacionar PIB municipal e vacinação |


Consolidação dos dados da camada Silver para análises com o objetivo de questionamentos. Coberturas vacinais por tipo de dose e regiões geográficas com periodo. Complementar os dados com o PIB, avaliando as condições econômicas.

In [0]:
#carga de dados
from pyspark.sql import functions as F

# Cobertura vacinal - Município
tvd1_mun = spark.table("mvp_1.silver.tvd1_municipio")
tvd2_mun = spark.table("mvp_1.silver.tvd2_municipio")
scrvz_mun = spark.table("mvp_1.silver.scrvz_municipio")

# Cobertura vacinal - UF
tvd1_uf = spark.table("mvp_1.silver.tvd1_uf")
tvd2_uf = spark.table("mvp_1.silver.tvd2_uf")
scrvz_uf = spark.table("mvp_1.silver.scrvz_uf")

# Cobertura vacinal - Região
tvd1_regiao = spark.table("mvp_1.silver.tvd1_regiao")
tvd2_regiao = spark.table("mvp_1.silver.tvd2_regiao")
scrvz_regiao = spark.table("mvp_1.silver.scrvz_regiao")

# PIB municipal
pib_mun = spark.table("mvp_1.silver.pib_municipio")

print("Tabelas Silver carregadas com sucesso.")

Tabelas Silver carregadas com sucesso.


In [0]:
print("=== TVD1 MUNICÍPIO ===")
tvd1_mun.printSchema()

print("=== TVD2 MUNICÍPIO ===")
tvd2_mun.printSchema()

print("=== SCRVZ MUNICÍPIO ===")
scrvz_mun.printSchema()

print("=== PIB MUNICÍPIO ===")
pib_mun.printSchema()

=== TVD1 MUNICÍPIO ===
root
 |-- codigo_municipio: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- codigo_uf: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- cobertura: double (nullable = true)

=== TVD2 MUNICÍPIO ===
root
 |-- codigo_municipio: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- codigo_uf: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- cobertura: double (nullable = true)

=== SCRVZ MUNICÍPIO ===
root
 |-- codigo_municipio: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- codigo_uf: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- cobertura: double (nullable = true)

=== PIB MUNICÍPIO ===
root
 |-- codigo_municipio_ibge: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- pib_mil_reais: double (nullable = true)



No datasus verifica-se um código de 6 dígitos, já no do IBGE ele traz o código de 7 dígitos.
"Para identificar os Entes da Federação, o SIOPS utiliza o código do IBGE,
com uma pequena particularidade, o código do IBGE é composto por sete dígitos e
o SIOPS utiliza apenas os seis primeiros dígitos. Ex.: o código do Município do Rio
de Janeiro é 3304557, no SIOPS é: 330455"

In [0]:
# Cria uma chave de 6 dígitos compatível com o código utilizado no DATASUS
pib_mun = (
    pib_mun
    .withColumn(
        "codigo_municipio",
        F.substring(F.col("codigo_municipio_ibge"), 1, 6)
    )
)

display(
    pib_mun.select(
        "codigo_municipio_ibge",
        "codigo_municipio",
        "municipio",
        "uf",
        "ano",
        "pib_mil_reais"
    ).limit(20)
)

codigo_municipio_ibge,codigo_municipio,municipio,uf,ano,pib_mil_reais
1100015,110001,Alta Floresta D'Oeste,RO,2002,111291.0
1100023,110002,Ariquemes,RO,2002,449593.0
1100031,110003,Cabixi,RO,2002,31768.0
1100049,110004,Cacoal,RO,2002,474443.0
1100056,110005,Cerejeiras,RO,2002,79174.0
1100064,110006,Colorado do Oeste,RO,2002,87254.0
1100072,110007,Corumbiara,RO,2002,45165.0
1100080,110008,Costa Marques,RO,2002,37308.0
1100098,110009,Espigão D'Oeste,RO,2002,119312.0
1100106,110010,Guajará-Mirim,RO,2002,174680.0


In [0]:
#Validação dos Códigos municipais distintos de cada fonte
codigos_pib = (
    pib_mun
    .select("codigo_municipio", "municipio")
    .distinct()
)

codigos_datasus = (
    tvd1_mun
    .select("codigo_municipio", "municipio")
    .distinct()
)

print(f"Municípios distintos no PIB: {codigos_pib.count()}")
print(f"Municípios distintos no DATASUS (TVD1): {codigos_datasus.count()}")

Municípios distintos no PIB: 5570
Municípios distintos no DATASUS (TVD1): 5571


O TVD1 possui aquele município extinto encontrado nas análises anteriores: 431453 – PINTO BANDEIRA (EXTINTO)
Transformar as tabelas municipais, regionais e federais em uma só, preservando a série histórica disponível

Validação da chave de integração entre DATASUS e IBGE

A base de PIB utiliza o código municipal completo do IBGE, composto por sete dígitos, enquanto a base de cobertura vacinal utiliza uma identificação municipal de seis dígitos.

A correspondência entre as fontes foi validada antes da realização do join. Todos os 5.570 municípios presentes na base de PIB encontraram tiveram um match com a da cobertura. O datasus apresentou um código adicional, referente a 431453 – PINTO BANDEIRA (EXTINTO). Esse registro não possui correspondência na base atual de PIB e não será artificialmente associado a outro município.

In [0]:
# Renomeia as coberturas para identificar cada indicador
tvd1_mun_gold = (
    tvd1_mun
    .withColumnRenamed("cobertura", "cobertura_tvd1")
)

tvd2_mun_gold = (
    tvd2_mun
    .withColumnRenamed("cobertura", "cobertura_tvd2")
)

scrvz_mun_gold = (
    scrvz_mun
    .withColumnRenamed("cobertura", "cobertura_scrvz")
)


In [0]:
#consolida num dataframe
cobertura_municipio = (
    tvd1_mun_gold.alias("d1")
    .join(
        tvd2_mun_gold.alias("d2"),
        ["codigo_municipio", "ano"],
        "full"
    )
    .join(
        scrvz_mun_gold.alias("sv"),
        ["codigo_municipio", "ano"],
        "full"
    )
)

In [0]:
# Excluindo colunas
cobertura_municipio = (
    cobertura_municipio
    .select(
        F.col("codigo_municipio"),
        
        F.coalesce(
            F.col("d1.municipio"),
            F.col("d2.municipio"),
            F.col("sv.municipio")
        ).alias("municipio"),
        
        F.coalesce(
            F.col("d1.codigo_uf"),
            F.col("d2.codigo_uf"),
            F.col("sv.codigo_uf")
        ).alias("codigo_uf"),
        
        F.col("ano"),
        F.col("d1.cobertura_tvd1"),
        F.col("d2.cobertura_tvd2"),
        F.col("sv.cobertura_scrvz")
    )
)

display(cobertura_municipio.limit(30))

codigo_municipio,municipio,codigo_uf,ano,cobertura_tvd1,cobertura_tvd2,cobertura_scrvz
110040,ALTO PARAISO,11,1999,0.0,null,null
110045,BURITIS,11,1999,0.0,null,null
110092,CHUPINGUAIA,11,1999,0.0,null,null
110175,VALE DO ANARI,11,1999,0.0,null,null
110180,VALE DO PARAISO,11,1999,0.0,null,null
120038,PLACIDO DE CASTRO,12,1999,0.0,null,null
130110,CAREIRO,13,1999,0.0,null,null
130370,SANTO ANTONIO DO ICA,13,1999,0.0,null,null
150110,BAGRE,15,1999,0.0,null,null
150277,CURIONOPOLIS,15,1999,0.0,null,null


In [0]:
#validação das colunas
from pyspark.sql import functions as F

total_registros = cobertura_municipio.count()

municipios = (
    cobertura_municipio
    .select("codigo_municipio")
    .distinct()
    .count()
)

duplicados = (
    cobertura_municipio
    .groupBy("codigo_municipio", "ano")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

periodo = (
    cobertura_municipio
    .agg(
        F.min("ano").alias("inicio"),
        F.max("ano").alias("fim")
    )
    .first()
)

print(f"Total de registros: {total_registros}")
print(f"Municípios distintos: {municipios}")
print(f"Duplicidades Município/Ano: {duplicados}")
print(f"Período: {periodo['inicio']}-{periodo['fim']}")

Total de registros: 133704
Municípios distintos: 5571
Duplicidades Município/Ano: 0
Período: 1999-2022


In [0]:
#Utilizando left join para ter os resultados de 1999 a 2001
pib_gold = (
    pib_mun
    .select(
        "codigo_municipio",
        "ano",
        "pib_mil_reais"
    )
)

gold_municipio = (
    cobertura_municipio.alias("vac")
    .join(
        pib_gold.alias("pib"),
        on=["codigo_municipio", "ano"],
        how="left"
    )
)

display(gold_municipio.limit(30))

codigo_municipio,ano,municipio,codigo_uf,cobertura_tvd1,cobertura_tvd2,cobertura_scrvz,pib_mil_reais
110040,1999,ALTO PARAISO,11,0.0,null,null,null
110045,1999,BURITIS,11,0.0,null,null,null
110092,1999,CHUPINGUAIA,11,0.0,null,null,null
110175,1999,VALE DO ANARI,11,0.0,null,null,null
110180,1999,VALE DO PARAISO,11,0.0,null,null,null
120038,1999,PLACIDO DE CASTRO,12,0.0,null,null,null
130110,1999,CAREIRO,13,0.0,null,null,null
130370,1999,SANTO ANTONIO DO ICA,13,0.0,null,null,null
150110,1999,BAGRE,15,0.0,null,null,null
150277,1999,CURIONOPOLIS,15,0.0,null,null,null


In [0]:
#Verificar se PIB está correto
#Validação do cruzamento entre cobertura vacinal e PIB

validacao_pib = (
    gold_municipio
    .filter(F.col("ano").between(2002, 2022))
    .agg(
        F.count("*").alias("total_registros"),
        F.count("pib_mil_reais").alias("com_pib"),
        F.sum(
            F.when(F.col("pib_mil_reais").isNull(), 1).otherwise(0)
        ).alias("sem_pib")
    )
)

display(validacao_pib)

total_registros,com_pib,sem_pib
116991,116896,95


Foi identificado 95 casos sem PIB, queremos entender essas linhas

In [0]:
#testes para achar os casos sem PIB
sem_pib = (
    gold_municipio
    .filter(
        F.col("ano").between(2002, 2022) &
        F.col("pib_mil_reais").isNull()
    )
    .select(
        "codigo_municipio",
        "municipio",
        "codigo_uf",
        "ano"
    )
    .orderBy("codigo_municipio", "ano")
)

display(sem_pib)

codigo_municipio,municipio,codigo_uf,ano
150475,MOJUI DOS CAMPOS,15,2002
150475,MOJUI DOS CAMPOS,15,2003
150475,MOJUI DOS CAMPOS,15,2004
150475,MOJUI DOS CAMPOS,15,2005
150475,MOJUI DOS CAMPOS,15,2006
150475,MOJUI DOS CAMPOS,15,2007
150475,MOJUI DOS CAMPOS,15,2008
150475,MOJUI DOS CAMPOS,15,2009
150475,MOJUI DOS CAMPOS,15,2010
150475,MOJUI DOS CAMPOS,15,2011


In [0]:
#Diferencia ausência de correspondência de PIB originalmente ausente
pib_diagnostico = (
    pib_mun
    .select(
        "codigo_municipio",
        "ano",
        "pib_mil_reais"
    )
    .withColumn("registro_pib_existe", F.lit(1))
)

diagnostico_sem_pib = (
    sem_pib.alias("vac")
    .join(
        pib_diagnostico.alias("pib"),
        on=["codigo_municipio", "ano"],
        how="left"
    )
    .withColumn(
        "motivo",
        F.when(
            F.col("registro_pib_existe").isNull(),
            "Município/Ano não encontrado na base do PIB"
        )
        .when(
            F.col("pib_mil_reais").isNull(),
            "Registro existe, mas PIB está ausente na fonte"
        )
        .otherwise("PIB disponível")
    )
)

display(
    diagnostico_sem_pib
    .groupBy("motivo")
    .count()
)

motivo,count
"Registro existe, mas PIB está ausente na fonte",74
Município/Ano não encontrado na base do PIB,21


Validação da integração com os dados de PIB

A integração entre os indicadores de cobertura vacinal e o PIB municipal foi realizada por código do município e ano. Para preservar todo o histórico de vacinação disponível, foi utilizado um left join, mantendo a cobertura vacinal como conjunto principal.

No período comum às duas fontes (2002–2022), foram identificados 95 registros sem valor de PIB. A análise desses casos demonstrou que 74 correspondem a registros existentes na base econômica cujo PIB está ausente na própria fonte, enquanto 21 correspondem a combinações Município/Ano não encontradas na base de PIB. Esses valores foram mantidos como nulos, sem imputação, preservando as características das fontes originais.

In [0]:
# Validação final da tabela GOLD

total_gold = gold_municipio.count()

duplicados_gold = (
    gold_municipio
    .groupBy("codigo_municipio", "ano")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Total de registros da Gold: {total_gold}")
print(f"Duplicidades Município/Ano: {duplicados_gold}")

Total de registros da Gold: 133704
Duplicidades Município/Ano: 0


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp_1.gold")
(
    gold_municipio
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_1.gold.indicadores_municipio")
)

print("Tabela mvp_1.gold.indicadores_municipio salva com sucesso.")

Tabela mvp_1.gold.indicadores_municipio salva com sucesso.


In [0]:
display(spark.sql("SHOW TABLES IN mvp_1.gold"))

database,tableName,isTemporary
gold,indicadores_municipio,false


In [0]:
tvd1_uf = spark.table("mvp_1.silver.tvd1_uf")
tvd2_uf = spark.table("mvp_1.silver.tvd2_uf")
scrvz_uf = spark.table("mvp_1.silver.scrvz_uf")

print("Tabelas de UF carregadas.")

Tabelas de UF carregadas.


In [0]:
cobertura_uf = (
    tvd1_uf.alias("d1")
    .join(
        tvd2_uf.alias("d2"),
        on=["codigo_uf", "ano"],
        how="left"
    )
    .join(
        scrvz_uf.alias("sv"),
        on=["codigo_uf", "ano"],
        how="left"
    )
    .select(
        F.col("codigo_uf"),
        F.col("d1.uf").alias("uf"),
        F.col("ano"),
        F.col("d1.cobertura").alias("cobertura_tvd1"),
        F.col("d2.cobertura").alias("cobertura_tvd2"),
        F.col("sv.cobertura").alias("cobertura_scrvz")
    )
)

display(cobertura_uf.limit(30))

codigo_uf,uf,ano,cobertura_tvd1,cobertura_tvd2,cobertura_scrvz
11,Rondônia,1999,0.0,null,null
12,Acre,1999,0.0,null,null
13,Amazonas,1999,0.0,null,null
14,Roraima,1999,0.0,null,null
15,Pará,1999,0.0,null,null
16,Amapá,1999,0.0,null,null
17,Tocantins,1999,0.0,null,null
21,Maranhão,1999,0.0,null,null
22,Piauí,1999,0.0,null,null
23,Ceará,1999,0.0,null,null


In [0]:
# Validação da tabela Gold por UF

total_uf = cobertura_uf.count()

ufs = (
    cobertura_uf
    .select("codigo_uf")
    .distinct()
    .count()
)

duplicados_uf = (
    cobertura_uf
    .groupBy("codigo_uf", "ano")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

periodo_uf = (
    cobertura_uf
    .agg(
        F.min("ano").alias("inicio"),
        F.max("ano").alias("fim")
    )
    .first()
)

print(f"Total de registros: {total_uf}")
print(f"UFs distintas: {ufs}")
print(f"Duplicidades UF/Ano: {duplicados_uf}")
print(f"Período: {periodo_uf['inicio']}-{periodo_uf['fim']}")

Total de registros: 648
UFs distintas: 27
Duplicidades UF/Ano: 0
Período: 1999-2022


In [0]:
# Salva a tabela consolidada por UF na camada Gold

(
    cobertura_uf
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_1.gold.indicadores_uf")
)

print("Tabela mvp_1.gold.indicadores_uf salva com sucesso.")

Tabela mvp_1.gold.indicadores_uf salva com sucesso.


In [0]:
# Carrega as tabelas Silver de cobertura vacinal por Região

tvd1_reg = spark.table("mvp_1.silver.tvd1_regiao")
tvd2_reg = spark.table("mvp_1.silver.tvd2_regiao")
scrvz_reg = spark.table("mvp_1.silver.scrvz_regiao")

print("=== TVD1 REGIÃO ===")
tvd1_reg.printSchema()

print("=== TVD2 REGIÃO ===")
tvd2_reg.printSchema()

print("=== SCRVZ REGIÃO ===")
scrvz_reg.printSchema()

=== TVD1 REGIÃO ===
root
 |-- codigo_regiao: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- cobertura: double (nullable = true)

=== TVD2 REGIÃO ===
root
 |-- codigo_regiao: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- cobertura: double (nullable = true)

=== SCRVZ REGIÃO ===
root
 |-- codigo_regiao: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- cobertura: double (nullable = true)



In [0]:
# Consolida as três coberturas vacinais por Região e Ano

cobertura_regiao = (
    tvd1_reg.alias("d1")
    .join(
        tvd2_reg.alias("d2"),
        on=["codigo_regiao", "regiao", "ano"],
        how="left"
    )
    .join(
        scrvz_reg.alias("sv"),
        on=["codigo_regiao", "regiao", "ano"],
        how="left"
    )
    .select(
        F.col("codigo_regiao"),
        F.col("regiao"),
        F.col("ano"),
        F.col("d1.cobertura").alias("cobertura_tvd1"),
        F.col("d2.cobertura").alias("cobertura_tvd2"),
        F.col("sv.cobertura").alias("cobertura_scrvz")
    )
)

display(cobertura_regiao.limit(30))

codigo_regiao,regiao,ano,cobertura_tvd1,cobertura_tvd2,cobertura_scrvz
1,Norte,1999,0.0,null,null
2,Nordeste,1999,0.0,null,null
3,Sudeste,1999,0.0,null,null
4,Sul,1999,0.0,null,null
5,Centro-Oeste,1999,525.47,null,null
1,Norte,2000,12.07,null,null
2,Nordeste,2000,65.03,null,null
3,Sudeste,2000,97.54,null,null
4,Sul,2000,87.6,null,null
5,Centro-Oeste,2000,87.15,null,null


In [0]:
# Validação da tabela Gold por Região

total_regiao = cobertura_regiao.count()

regioes = (
    cobertura_regiao
    .select("codigo_regiao")
    .distinct()
    .count()
)

duplicados_regiao = (
    cobertura_regiao
    .groupBy("codigo_regiao", "ano")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

periodo_regiao = (
    cobertura_regiao
    .agg(
        F.min("ano").alias("inicio"),
        F.max("ano").alias("fim")
    )
    .first()
)

print(f"Total de registros: {total_regiao}")
print(f"Regiões distintas: {regioes}")
print(f"Duplicidades Região/Ano: {duplicados_regiao}")
print(f"Período: {periodo_regiao['inicio']}-{periodo_regiao['fim']}")

Total de registros: 120
Regiões distintas: 5
Duplicidades Região/Ano: 0
Período: 1999-2022


In [0]:
# Salva a tabela consolidada por Região na camada Gold

(
    cobertura_regiao
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_1.gold.indicadores_regiao")
)

print("Tabela mvp_1.gold.indicadores_regiao salva com sucesso.")

Tabela mvp_1.gold.indicadores_regiao salva com sucesso.


In [0]:
display(spark.sql("SHOW TABLES IN mvp_1.gold"))

database,tableName,isTemporary
gold,indicadores_municipio,false
gold,indicadores_regiao,false
gold,indicadores_uf,false
